# 9. Planeación de transporte

Una empresa debe transportar máquinas desde las plantas de producción A, B y C hasta los almacenes P , Q y S. Se requieren cinco máquinas en P , cuatro en Q y tres en S, mientras que hay ocho máquinas disponibles en A, cinco en B y tres en C. Los costos de transporte, en dólares, se proporcionan en la Tabla 8.

|**Planta / Almacén**|**P**|**Q**|**S**|
|---|---|---|---|
|A|50|60|30|
|B|60|40|20|
|C|40|70|30|

_Tabla 8: Costos de transporte, en dólares por máquina._

**a)** Formule un modelo de programación lineal entera que minimice los costos de transporte.

**b)** Suponga que el costo de transportar una máquina desde la planta B aumenta en $10 a
partir de la tercera máquina; es decir, para la cuarta, la quinta, etcétera. Reformule el
modelo del inciso anterior teniendo en cuenta este supuesto.

In [12]:
import numpy as np

costos = [
    # P   Q   S
    [50, 60, 30],  # A
    [60, 40, 20],  # B
    [40, 70, 30]   # C
]

costos_np = np.array(costos)

## Datos y supuestos  
* transportar máquinas desde las plantas a los almacenes
* DEMANDA: Se requieren cinco máquinas en P , cuatro en Q y tres en S
* OFERTA: Hay ocho máquinas disponibles en A, cinco en B y tres en C
* COSTOS de transporte en Tabla 8
* minimice los costos
* $COSTOS_{Bj}$ + 10 

In [13]:
from docplex.mp.model import Model

mdl = Model(name='Planeacion de transporte')

## Formulación matemática 
### Variables de decisión
* Variable entera
* Maquinas desde planta $i \epsilon \{A,B,C\}$ a almacén $j \epsilon \{P, Q, S\}$
$$x_{ij}$$

In [14]:
plantas_i = ['A', 'B', 'C']
almacenes_j = ['P', 'Q', 'S']

In [15]:
# Variable de decision
x = mdl.integer_var_matrix(plantas_i, almacenes_j, lb=0, name='x')

In [16]:
x

{('A', 'P'): docplex.mp.Var(type=I,name='x_A_P'),
 ('A', 'Q'): docplex.mp.Var(type=I,name='x_A_Q'),
 ('A', 'S'): docplex.mp.Var(type=I,name='x_A_S'),
 ('B', 'P'): docplex.mp.Var(type=I,name='x_B_P'),
 ('B', 'Q'): docplex.mp.Var(type=I,name='x_B_Q'),
 ('B', 'S'): docplex.mp.Var(type=I,name='x_B_S'),
 ('C', 'P'): docplex.mp.Var(type=I,name='x_C_P'),
 ('C', 'Q'): docplex.mp.Var(type=I,name='x_C_Q'),
 ('C', 'S'): docplex.mp.Var(type=I,name='x_C_S')}

In [17]:
x['A', 'P']

docplex.mp.Var(type=I,name='x_A_P')

### Restricciones

#### Restricciones 1: Demanda
* Se requieren cinco máquinas en P , cuatro en Q y tres en S
* == , pues es el numero de maquinas que se requieren, y no hay indicaciones para excedentes

$$\sum_i x_{iP} == 5$$
$$\sum_i x_{iQ} == 4$$
$$\sum_i x_{iS} == 3$$

In [ ]:
mdl.add_constraint(
	mdl.sum(x[i, 'P'] for i in plantas_i) == 5
)

mdl.add_constraint(
	mdl.sum(x[i, 'Q'] for i in plantas_i) == 4
)

mdl.add_constraint(
	mdl.sum(x[i, 'S'] for i in plantas_i) == 3
)

docplex.mp.LinearConstraint[](x_A_S+x_B_S+x_C_S,EQ,3)

#### Restricciones 2: Oferta
* Hay ocho máquinas disponibles en A, cinco en B y tres en C
* $\le$ , pues hay mas oferta que demanda

$$\sum_j x_{Aj} \le 8$$
$$\sum_j x_{Bj} \le 5$$
$$\sum_j x_{Cj} \le 3$$

In [19]:
mdl.add_constraint(
	mdl.sum(x['A', j] for j in almacenes_j) <= 8
)

mdl.add_constraint(
	mdl.sum(x['B', j] for j in almacenes_j) <= 5
)

mdl.add_constraint(
	mdl.sum(x['C', j] for j in almacenes_j) <= 3
)

docplex.mp.LinearConstraint[](x_C_P+x_C_Q+x_C_S,LE,3)

## Función objetivo 

$$\min z = 50 x_{AP} + 60x_{AQ} + 30x_{AS} + 60x_{BP} + 40x_{BQ} + 20x_{BS} + 40x_{CP} + 70x_{CQ} + 30x_{CS}$$


In [20]:
# Funcion objetivo
mdl.minimize(
    mdl.sum(costos_np[i_idx, j_idx] * x[i, j] 
            for i_idx, i in enumerate(plantas_i) 
            for j_idx, j in enumerate(almacenes_j))
)

In [21]:
# Resolver e imprimir solucion
solution = mdl.solve(log_output=True)

print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 530.000000 after 0.01 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 6 rows, 9 columns, and 18 nonzeros.
Reduced MIP has 0 binaries, 9 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.01 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 6 rows, 9 columns, and 18 nonzeros.
Reduced MIP has 0 binaries, 9 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.01 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 4 threads.
Root relaxation solution time = 0.00 sec. (0.01 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                          530.0000        0.0000           100.00%
*     0+    0               